# Meta-heurísticas - 2026/2
## Atividade T1

### Gerador de Instâncias



#### Lógica da criação de instâncias

1. Recebe o número de pessoas;

2. Define o tamanho do núcleo, ou seja, um subgrupo que soma zero e será distribuído entre diferentes grupos;

3. Adiciona valores ao núcleo, forçando a última pessoa a absorver a diferena para que a soma final seja zero;

4. Determina o número k de subgrupos sendo o tamanho do núcleo dividido por 2;

5. Distribui as demais pessoas entre os k subgrupos, de forma que o grupo tenha soma total igual a zero.

In [ ]:
import random
import os

def gerar_instancia_com_intersecao(n_pessoas, nome_arquivo):
    # Define o tamanho do núcleo (20% das pessoas, mínimo de 4)
    tamanho_nucleo = max(4, int(n_pessoas * 0.2))
    if tamanho_nucleo % 2 != 0:
        # Garante que seja par para facilitar a divisão
        tamanho_nucleo += 1 

    pessoas = list(range(1, n_pessoas + 1))
    random.shuffle(pessoas)

    nucleo = pessoas[:tamanho_nucleo]
    restantes = pessoas[tamanho_nucleo:]

    saldos = {p: 0 for p in pessoas}

    # Atribui saldos ao núcleo para que a soma de todos eles seja zero
    soma_nucleo = 0
    for p in nucleo[:-1]:
        valor = random.randint(-500, 500)
        while valor == 0:
            valor = random.randint(-500, 500)
        saldos[p] = valor
        soma_nucleo += valor

    # O último do núcleo absorve a diferença para zerar o grupo
    saldos[nucleo[-1]] = -soma_nucleo 

    # Divide o núcleo em pares (subgrupos)
    k_grupos = tamanho_nucleo // 2
    subgrupos_nucleo = [nucleo[i:i + 2] for i in range(0, tamanho_nucleo, 2)]

    # Divide as demais pessoas em k_grupos de forma equilibrada
    subgrupos_restantes = [[] for _ in range(k_grupos)]
    for i, p in enumerate(restantes):
        subgrupos_restantes[i % k_grupos].append(p)

    # Faz a interseção
    for i in range(k_grupos):
        soma_sub_nucleo = sum(saldos[p] for p in subgrupos_nucleo[i])
        grupo_restante = subgrupos_restantes[i]

        if not grupo_restante:
            continue

        soma_temp_restante = 0
        # Gera saldos
        for p in grupo_restante[:-1]:
            valor = random.randint(-500, 500)
            while valor == 0:
                valor = random.randint(-500, 500)
            saldos[p] = valor
            soma_temp_restante += valor

        saldos[grupo_restante[-1]] = -(soma_sub_nucleo + soma_temp_restante)

    # Remove quem ficou com zero e embaralha a lista final
    pessoas_info = [{"id": p, "saldo": saldos[p]} for p in pessoas if saldos[p] != 0]
    pessoas_info.sort(key=lambda x: x["id"])

    # Salva no arquivo
    with open(nome_arquivo, 'w') as f:
        for pessoa in pessoas_info:
            f.write(f"{pessoa['id']} {pessoa['saldo']}\n")
            
    # Salva os metadados (para analisar a qualidade da sua busca local depois)
    os.makedirs("data/metadados", exist_ok=True)
    nome_base = os.path.basename(nome_arquivo)
    nome_arquivo_meta = f"data/metadados/{nome_base.replace('.txt', '_metadados.txt')}"
    
    with open(nome_arquivo_meta, 'w') as f:
        f.write("=== METADADOS (GABARITO DA INSTÂNCIA) ===\n")
        f.write(f"Tamanho Núcleo: {tamanho_nucleo} | Quantidade de Grupos Ótimos: {k_grupos}\n")
        f.write(f"-> O algoritmo Guloso tende a errar gerando {n_pessoas - 2} transações.\n")
        f.write(f"-> O Ótimo Global plantado é de {n_pessoas - k_grupos} transações.\n")
        f.write("=========================================\n")

- Determina o tamanho das instâncias
- Gera as instâncias chamando o método de geração de instâncias

In [2]:
valores = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 460, 480, 500]
random.seed(42)

os.makedirs("data/instancias", exist_ok=True)

for numeros in valores:
    caminho_arquivo = f"data/instancias/instancia_splitwise_{numeros}.txt"
    gerar_instancia_com_intersecao(numeros, caminho_arquivo)

### Leitura dos arquivos

- Determinamos os tamanhos/prefixos dos arquivos;
- Lemos todas as instâncias.

In [148]:
import pandas as pd
import time

def ler_instancia_para_df(nome_arquivo, n_pessoas):
    df = pd.read_csv(nome_arquivo, sep=" ", names=["id", "saldo"])
    return df

In [149]:
instancias = {}
tamanhos = [
    "9", "20", "30", "50", "50_8", "100", "100_20", "150", "150_15",
    "200", "200_8", "200_80", "300", "300_60", "400", "400_35", "500",
    "500_25", "500_188", "600", "600_40", "700", "700_90", "800", "800_80",
    "900", "900_100", "1000", "1000_120", "1100", "1300", "1500"
]

for tamanho in tamanhos:
    caminho_arquivo = f"data/instancias-turma/instancia_splitwise_{tamanho}.txt"
    df_carregado = ler_instancia_para_df(caminho_arquivo, tamanho)

    instancias[tamanho] = df_carregado

### Iniciando busca de solução

#### **Algoritmo Construtivo**

##### **Lógica do algoritmo construtivo guloso**

1. Separa as pessoas em duas listas: credores (saldo positivo) e devedores (saldo negativo);

2. Ordena as duas listas a cada iteração, garantindo que os maiores devedores e os maiores credores fiquem sempre no topo;

3. Tenta primeiro um "casamento perfeito": verifica se o valor da dívida de algum devedor é exatamente igual ao crédito de algum credor, zerando os dois de uma vez;

4. Caso não haja um valor exato, faz a escolha gulosa: junta o maior devedor com o maior credor global, transferindo o máximo de valor possível entre eles;

5. Atualiza os saldos resultantes, remove da lista quem zerou a conta, e repete o processo até que não sobrem mais dívidas.

In [150]:
def construtivo_guloso(df_instancia):
    df_temp = df_instancia.copy()
    inicio_tempo = time.perf_counter()

    credores = df_temp[df_temp['saldo'] > 0].values.tolist()
    devedores = df_temp[df_temp['saldo'] < 0].values.tolist()
    transacoes = []

    while credores and devedores:
        devedores.sort(key=lambda x: abs(x[1]), reverse=True)
        credores.sort(key=lambda x: x[1], reverse=True)

        match_encontrado = False

        mapa_credores = {cred[1]: j for j, cred in enumerate(credores)}

        for i, dev in enumerate(devedores):
            saldo_pendente = abs(dev[1])

            if saldo_pendente in mapa_credores:
                j = mapa_credores[saldo_pendente]
                cred = credores[j]

                transacoes.append({
                    'devedor': int(dev[0]),
                    'credor': int(cred[0]),
                    'valor': cred[1]
                })

                devedores.pop(i)
                credores.pop(j)
                match_encontrado = True
                break

        if not match_encontrado:
            dev = devedores[0]
            cred = credores[0]

            valor_transacao = min(abs(dev[1]), cred[1])

            transacoes.append({
                'devedor': int(dev[0]),
                'credor': int(cred[0]),
                'valor': valor_transacao
            })

            devedores[0][1] += valor_transacao
            credores[0][1] -= valor_transacao

            if devedores[0][1] == 0:
                devedores.pop(0)
            if credores[0][1] == 0:
                credores.pop(0)

    fim_tempo = time.perf_counter()
    tempo_execucao = fim_tempo - inicio_tempo

    return transacoes, tempo_execucao

##### **Lógica do algoritmo construtivo randomizado**
1. Separa as pessoas em duas listas: credores (saldo positivo) e devedores (saldo negativo);

2. Ordena as listas a cada iteração, mantendo os maiores devedores e os maiores credores no topo;

3. Usa o parâmetro alpha para limitar as opções (no nosso caso, os top 20%) e sorteia aleatoriamente um devedor dentro dessa restrição;

4. Tenta primeiro um "casamento perfeito": procura se existe algum credor com o valor exato da dívida desse devedor sorteado;

5. Caso não encontre o valor exato, usa novamente o alpha para limitar o grupo dos maiores credores e sorteia aleatoriamente um credor entre eles;

6. Transfere o máximo de valor possível entre os dois sorteados, atualiza os saldos, remove quem zerou a conta e repete o processo.

In [151]:
def construtivo_randomizado(df_instancia, alpha=0.2):
    df_temp = df_instancia.copy()
    inicio_tempo = time.perf_counter()

    credores = df_temp[df_temp['saldo'] > 0].values.tolist()
    devedores = df_temp[df_temp['saldo'] < 0].values.tolist()

    transacoes = []

    while credores and devedores:
        devedores.sort(key=lambda x: x[1])
        credores.sort(key=lambda x: x[1], reverse=True)

        limite_dev = max(1, int(alpha * len(devedores)))
        idx_dev = random.randrange(limite_dev)
        dev = devedores[idx_dev]
        magnitude_dev = -dev[1]

        idx_cred = -1
        for j, cred in enumerate(credores):
            if cred[1] == magnitude_dev:
                idx_cred = j
                break

        if idx_cred == -1:
            limite_cred = max(1, int(alpha * len(credores)))
            idx_cred = random.randrange(limite_cred)

        cred = credores[idx_cred]

        valor_transacao = min(magnitude_dev, cred[1])
        transacoes.append({
            'devedor': int(dev[0]),
            'credor': int(cred[0]),
            'valor': valor_transacao
        })

        devedores[idx_dev][1] += valor_transacao
        credores[idx_cred][1] -= valor_transacao

        if devedores[idx_dev][1] == 0:
            devedores.pop(idx_dev)
        if credores[idx_cred][1] == 0:
            credores.pop(idx_cred)


    fim_tempo = time.perf_counter()
    tempo_execucao = fim_tempo - inicio_tempo

    return transacoes, tempo_execucao

#### Algoritmo Busca Local

#### Lógica da vizihança: destruição e reconstrução

1. Conta a frequência de cada pessoa nas transações e ordena a lista, colocando no topo aqueles que fazem muitas transferências (indicando ineficiência);

2. Sorteia os "alvos" da destruição escolhidos entre os piores da lista (pegamos aleatoriamente 20% dentre os 40% com maior frequência de transações);

3. Destruição: Rompe todas as transações que envolvem essas pessoas sorteadas;

4. Calcula os saldos dos indivíduos após desfazer as transações;

5. Reconstrução: Aplica o algoritmo construtivo (Guloso ou Randomizado) exclusivamente entre as transações que foram rompidas;

6. Critério de Aceitação: Se a reconstrução resolver as dívidas desse grupo com menos transações do que havia antes, a melhoria é aceita e mesclada ao resto da solução. Caso contrário, a alteração é descartada.

##### Funções Auxiliares

In [ ]:
def vizinhanca_destruicao_reconstrucao(solucao_atual, tipo, taxa_destruicao=0.2):
    # Conta a frequência de cada pessoa nas transações
    frequencia = {}
    for t in solucao_atual:
        frequencia[t['devedor']] = frequencia.get(t['devedor'], 0) + 1
        frequencia[t['credor']] = frequencia.get(t['credor'], 0) + 1

    # Ordena pessoas pelo número de transações (do maior para o menor)
    pessoas_ordenadas = sorted(frequencia.keys(), key=lambda x: frequencia[x], reverse=True)

    # Seleciona o alvo com um leve grau de aleatoriedade para não estagnar
    qtd_selecionar = max(3, int(len(pessoas_ordenadas) * taxa_destruicao))
    
    # Pegamos os 40% piores e sorteamos 'qtd_selecionar' pessoas de dentro desse grupo
    pool_candidatos = pessoas_ordenadas[:max(qtd_selecionar * 2, 4)]
    pessoas_alvo = set(random.sample(pool_candidatos, min(qtd_selecionar, len(pool_candidatos))))

    # Separa as transações que envolvem essas pessoas
    transacoes_restantes = []
    transacoes_destruidas = []
    
    for t in solucao_atual:
        # Se qualquer uma das pontas da transação for um alvo, ela é destruída
        if t['devedor'] in pessoas_alvo or t['credor'] in pessoas_alvo:
            transacoes_destruidas.append(t)
        else:
            transacoes_restantes.append(t)

    # Calcular o saldo LOCAL exato a partir das transações rompidas
    saldos_locais = {}
    for t in transacoes_destruidas:
        saldos_locais[t['devedor']] = saldos_locais.get(t['devedor'], 0) - t['valor']
        saldos_locais[t['credor']] = saldos_locais.get(t['credor'], 0) + t['valor']

    # Criar um DataFrame no mesmo formato que a função gulosa espera
    df_local = pd.DataFrame(
        [{'id': p, 'saldo': s} for p, s in saldos_locais.items() if s != 0]
    )

    if df_local.empty:
        return solucao_atual
        
    # Reconstrói usando a função já existente
    df_local.columns = ['id', 'saldo'] 

    if tipo == "random":
        novas_transacoes, _ = construtivo_randomizado(df_local, alpha=0.8)
    else:
        novas_transacoes, _ = construtivo_guloso(df_local)

    # Avaliação
    if len(novas_transacoes) < len(transacoes_destruidas):
        # Houve melhoria
        return transacoes_restantes + novas_transacoes
    else:
        # Não houve melhoria, devolvemos a solução intacta
        return solucao_atual

##### **Lógica do algoritmo de busca local - first improvement**

1. Inicia a partir de uma solução base (fornecida pelo algoritmo construtivo) e define um limite de tentativas consecutivas sem sucesso como critério de parada;

2. Gera um único vizinho através do método de destruição e reconstrução;

3. Critério First-Improvement: Avalia esse vizinho imediatamente. Se a nova solução possuir menos transações que a atual, ela é aceita na mesma hora como a nova solução vigente, e o contador de tentativas falhas é zerado;

4. Caso a nova solução não seja melhor, ela é descartada e o contador de falhas é incrementado;

> O processo se repete até que o algoritmo atinja o limite máximo de tentativas seguidas sem encontrar nenhuma melhoria, retornando a solução final, o tempo gasto e o número total de iterações.

In [ ]:
import time

def bl_first_improvement(solucao_inicial, tipo, max_tentativas=200):
    solucao_atual = solucao_inicial.copy()
    inicio_tempo = time.perf_counter()
    
    n_it = 0 # Contador de iterações
    tentativas_sem_melhora = 0
    
    while tentativas_sem_melhora < max_tentativas:
        n_it += 1
        
        # Gera UM vizinho aplicando o movimento de destruição e reconstrução
        vizinho = vizinhanca_destruicao_reconstrucao(solucao_atual, tipo)
        
        # Critério First-Improvement: Aceita a primeira melhora imediata
        if len(vizinho) < len(solucao_atual):
            solucao_atual = vizinho
            tentativas_sem_melhora = 0 # Zera o contador, pois achou uma melhora
        else:
            tentativas_sem_melhora += 1 # Não melhorou, incrementa o contador de falhas
            
    tempo_total = time.perf_counter() - inicio_tempo
    
    return solucao_atual, tempo_total, n_it

##### **Lógica do algoritmo de busca local - best improvement**

1. Inicia a partir de uma solução base e define um limite de tentativas consecutivas sem sucesso, além de calcular previamente o tamanho da "vizinhança" (quantos vizinhos serão testados por ciclo - no nosso exemplo usamos 10% do tamanho da instância);

2. Exploração em Lote: Em vez de olhar apenas um vizinho por vez, o algoritmo gera múltiplos vizinhos em sequência usando o método de destruição e reconstrução;

3. Avalia todas as soluções geradas nesse lote internamente para identificar qual delas possui a maior redução no número de transações;

4. Critério Best-Improvement: Compara apenas a melhor solução encontrada no lote com a solução atual. Se ela for melhor, é aceita como a nova solução vigente e o contador de falhas é zerado;

5. Caso nenhum vizinho explorado no lote consiga superar a solução atual, o contador de falhas é incrementado;

> O processo se repete até que o algoritmo atinja o limite máximo de iterações consecutivas sem encontrar nenhuma melhoria global, retornando o resultado final.

In [ ]:
import time

def bl_best_improvement(solucao_inicial, tipo, max_tentativas=200):
    solucao_atual = solucao_inicial.copy()
    tamanho_vizinhanca = max(10, int(len(solucao_atual) * 0.1))
    inicio_tempo = time.perf_counter()
    
    n_it = 0 
    tentativas_sem_melhora = 0
    
    while tentativas_sem_melhora < max_tentativas:
        n_it += 1
        
        melhor_vizinho_local = None
        menor_tamanho_local = len(solucao_atual)
        
        # Explora a vizinhança: gera múltiplos vizinhos antes de tomar uma decisão
        for _ in range(tamanho_vizinhanca):
            vizinho = vizinhanca_destruicao_reconstrucao(solucao_atual, tipo)
            
            if len(vizinho) < menor_tamanho_local:
                menor_tamanho_local = len(vizinho)
                melhor_vizinho_local = vizinho
        
        # Critério Best-Improvement: Aceita apenas o melhor vizinho do lote, se houver melhora
        if melhor_vizinho_local is not None:
            solucao_atual = melhor_vizinho_local
            tentativas_sem_melhora = 0 
        else:
            tentativas_sem_melhora += 1 
            
    tempo_total = time.perf_counter() - inicio_tempo
    
    return solucao_atual, tempo_total, n_it

### RESULTADOS FINAIS

In [ ]:
resultados_tabela = []

for tamanho, df_inst in instancias.items():
    # Executa Algoritmos Construtivos
    transacoes_g, tempo_g = construtivo_guloso(df_inst)
    transacoes_r, tempo_r = construtivo_randomizado(df_inst)

    # Executa First-Improvement sobre as duas soluções iniciais
    sol_fi_g, t_fi_g, nit_fi_g = bl_first_improvement(transacoes_g, "guloso")
    sol_fi_r, t_fi_r, nit_fi_r = bl_first_improvement(transacoes_r, "random")

    # Executa Best-Improvement sobre as duas soluções iniciais
    sol_bi_g, t_bi_g, nit_bi_g = bl_best_improvement(transacoes_g, "guloso")
    sol_bi_r, t_bi_r, nit_bi_r = bl_best_improvement(transacoes_r, "random")
    print(f"Finalizando instancias {tamanho}!")

    # Grava os resultados consolidados
    resultados_tabela.append({
        'I': f"Inst_{tamanho}",

        'AC_G (sol)': len(transacoes_g),
        'AC_G (t(s))': round(tempo_g, 2),
        'AC_R (sol)': len(transacoes_r),
        'AC_R (t(s))': round(tempo_r, 2),

        'BL_FI(AC_G) (sol)': len(sol_fi_g),
        'BL_FI(AC_G) (t(s))': round(t_fi_g, 2),
        'BL_FI(AC_G) (N_It)': nit_fi_g,

        'BL_FI(AC_R) (sol)': len(sol_fi_r),
        'BL_FI(AC_R) (t(s))': round(t_fi_r, 2),
        'BL_FI(AC_R) (N_It)': nit_fi_r,

        'BL_BI(AC_G) (sol)': len(sol_bi_g),
        'BL_BI(AC_G) (t(s))': round(t_bi_g, 2),
        'BL_BI(AC_G) (N_It)': nit_bi_g,

        'BL_BI(AC_R) (sol)': len(sol_bi_r),
        'BL_BI(AC_R) (t(s))': round(t_bi_r, 2),
        'BL_BI(AC_R) (N_It)': nit_bi_r
    })

df_resultados_parciais = pd.DataFrame(resultados_tabela)
print(df_resultados_parciais.to_string())

Finalizando instancias 9!
Finalizando instancias 20!
Finalizando instancias 30!
Finalizando instancias 50!
Finalizando instancias 50_8!
Finalizando instancias 100!
Finalizando instancias 100_20!
Finalizando instancias 150!
Finalizando instancias 150_15!
Finalizando instancias 200!
Finalizando instancias 200_8!
Finalizando instancias 200_80!
Finalizando instancias 300!
Finalizando instancias 300_60!
Finalizando instancias 400!
Finalizando instancias 400_35!
Finalizando instancias 500!
Finalizando instancias 500_25!
Finalizando instancias 500_188!
Finalizando instancias 600!
Finalizando instancias 600_40!
Finalizando instancias 700!
Finalizando instancias 700_90!
Finalizando instancias 800!
Finalizando instancias 800_80!
Finalizando instancias 900!
Finalizando instancias 900_100!
Finalizando instancias 1000!
Finalizando instancias 1000_120!
Finalizando instancias 1100!


In [ ]:
import os

os.makedirs("output", exist_ok=True)
df_resultados_parciais.to_csv("output/resultados_1.csv", index=False)
with open("output/resultados_1.txt", "w") as f:
    f.write(df_resultados_parciais.to_string())

In [ ]:
import pandas as pd

# Define a estrutura do cabeçalho de múltiplos níveis (MultiIndex)
colunas_multi = pd.MultiIndex.from_tuples([
    ('AC_G', 'sol'), ('AC_G', 't(s)'),
    ('AC_R', 'sol'), ('AC_R', 't(s)'),
    ('BL_FI(AC_G)', 'sol'), ('BL_FI(AC_G)', 'N_It'), ('BL_FI(AC_G)', 't(s)'),
    ('BL_FI(AC_R)', 'sol'), ('BL_FI(AC_R)', 'N_It'), ('BL_FI(AC_R)', 't(s)'),
    ('BL_BI(AC_G)', 'sol'), ('BL_BI(AC_G)', 'N_It'), ('BL_BI(AC_G)', 't(s)'),
    ('BL_BI(AC_R)', 'sol'), ('BL_BI(AC_R)', 'N_It'), ('BL_BI(AC_R)', 't(s)')
])

# Mapeia as colunas originais geradas pelo seu código na ordem correta do MultiIndex
colunas_originais = [
    'AC_G (sol)', 'AC_G (t(s))',
    'AC_R (sol)', 'AC_R (t(s))',
    'BL_FI(AC_G) (sol)', 'BL_FI(AC_G) (N_It)', 'BL_FI(AC_G) (t(s))',
    'BL_FI(AC_R) (sol)', 'BL_FI(AC_R) (N_It)', 'BL_FI(AC_R) (t(s))',
    'BL_BI(AC_G) (sol)', 'BL_BI(AC_G) (N_It)', 'BL_BI(AC_G) (t(s))',
    'BL_BI(AC_R) (sol)', 'BL_BI(AC_R) (N_It)', 'BL_BI(AC_R) (t(s))'
]

# Cria o novo DataFrame estruturado
df_tabela = df_resultados_parciais[colunas_originais].copy()
df_tabela.columns = colunas_multi
df_tabela.index = df_resultados_parciais['I']
df_tabela.index.name = 'I'

# Função para destacar a negrito o menor valor de sol e t(s) em cada linha
def destacar_melhores(linha):
    estilos = [''] * len(linha)
    
    idx_sol = [i for i, col in enumerate(linha.index) if col[1] == 'sol']
    idx_t = [i for i, col in enumerate(linha.index) if col[1] == 't(s)']
    
    if idx_sol:
        min_sol = min([linha.iloc[i] for i in idx_sol])
        for i in idx_sol:
            if linha.iloc[i] == min_sol:
                estilos[i] = 'font-weight: bold; color: black;'
                
    return estilos

# Seleciona apenas as colunas de tempo para aplicar a formatação de 2 casas
colunas_tempo = [col for col in df_tabela.columns if col[1] == 't(s)']

# Renderiza a tabela com formatação decimal E negrito
tabela_formatada = (df_tabela.style
                    .format("{:.2f}", subset=colunas_tempo)
                    .apply(destacar_melhores, axis=1))

tabela_formatada

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from IPython.display import display 

# FORMATAÇÃO E EXIBIÇÃO DA TABELA

colunas_multi = pd.MultiIndex.from_tuples([
    ('AC_G', 'sol'), ('AC_G', 't(s)'),
    ('AC_R', 'sol'), ('AC_R', 't(s)'),
    ('BL_FI(AC_G)', 'sol'), ('BL_FI(AC_G)', 'N_It'), ('BL_FI(AC_G)', 't(s)'),
    ('BL_FI(AC_R)', 'sol'), ('BL_FI(AC_R)', 'N_It'), ('BL_FI(AC_R)', 't(s)'),
    ('BL_BI(AC_G)', 'sol'), ('BL_BI(AC_G)', 'N_It'), ('BL_BI(AC_G)', 't(s)'),
    ('BL_BI(AC_R)', 'sol'), ('BL_BI(AC_R)', 'N_It'), ('BL_BI(AC_R)', 't(s)')
])

colunas_originais = [
    'AC_G (sol)', 'AC_G (t(s))',
    'AC_R (sol)', 'AC_R (t(s))',
    'BL_FI(AC_G) (sol)', 'BL_FI(AC_G) (N_It)', 'BL_FI(AC_G) (t(s))',
    'BL_FI(AC_R) (sol)', 'BL_FI(AC_R) (N_It)', 'BL_FI(AC_R) (t(s))',
    'BL_BI(AC_G) (sol)', 'BL_BI(AC_G) (N_It)', 'BL_BI(AC_G) (t(s))',
    'BL_BI(AC_R) (sol)', 'BL_BI(AC_R) (N_It)', 'BL_BI(AC_R) (t(s))'
]

df_tabela = df_resultados_parciais[colunas_originais].copy()
df_tabela.columns = colunas_multi
df_tabela.index = df_resultados_parciais['I']
df_tabela.index.name = 'I'

def destacar_melhor_bl(linha):
    estilos = [''] * len(linha)
    
    # Isola apenas as colunas de solução (sol) das Buscas Locais
    colunas_bl_sol = [
        ('BL_FI(AC_G)', 'sol'), ('BL_FI(AC_R)', 'sol'),
        ('BL_BI(AC_G)', 'sol'), ('BL_BI(AC_R)', 'sol')
    ]
    
    idx_bl_sol = [i for i, col in enumerate(linha.index) if col in colunas_bl_sol]
    
    if idx_bl_sol:
        min_bl = min([linha.iloc[i] for i in idx_bl_sol])
        for i in idx_bl_sol:
            if linha.iloc[i] == min_bl:
                estilos[i] = 'font-weight: bold; color: black;'
                
    return estilos

colunas_tempo = [col for col in df_tabela.columns if col[1] == 't(s)']

total_instancias = len(df_tabela)
tamanho_chunk = total_instancias // 3 + (1 if total_instancias % 3 > 0 else 0)
tabelas_divididas = [df_tabela.iloc[i:i + tamanho_chunk] for i in range(0, total_instancias, tamanho_chunk)]

print("========== TABELAS DE RESULTADOS ==========\n")
for i, df_part in enumerate(tabelas_divididas, 1):
    tabela_formatada = (df_part.style
                        .format("{:.4f}", subset=colunas_tempo) # .4f evita zerar tempos muito rápidos
                        .apply(destacar_melhor_bl, axis=1)
                        .set_caption(f"Parte {i}"))
    
    # O display garante que a tabela desenhada no HTML do Colab apareça no output
    display(tabela_formatada)


# GERAÇÃO DOS GRÁFICOS DE GANHO

instancias_nomes = df_resultados_parciais['I'].tolist()

colunas_bl_g = ['BL_FI(AC_G) (sol)', 'BL_BI(AC_G) (sol)']
colunas_bl_r = ['BL_FI(AC_R) (sol)', 'BL_BI(AC_R) (sol)']

min_g = df_resultados_parciais[colunas_bl_g].min(axis=1)
min_r = df_resultados_parciais[colunas_bl_r].min(axis=1)

melhor_bl = np.where(min_g <= min_r, min_g, min_r)
origem_linhagem = np.where(min_g <= min_r, 'G', 'R')

ac_correspondente = np.where(origem_linhagem == 'G', 
                               df_resultados_parciais['AC_G (sol)'], 
                               df_resultados_parciais['AC_R (sol)'])

f_construtivo = ac_correspondente.tolist()
f_busca_local = melhor_bl.tolist()

os.makedirs("output", exist_ok=True)

print("\n========== GRÁFICOS DE EVOLUÇÃO ==========\n")
for i in range(3):
    inicio = i * tamanho_chunk
    fim = min((i + 1) * tamanho_chunk, total_instancias)
    
    if inicio >= total_instancias:
        break
        
    x_chunk = instancias_nomes[inicio:fim]
    y_constr_chunk = f_construtivo[inicio:fim]
    y_bl_chunk = f_busca_local[inicio:fim]
    
    fig, ax = plt.subplots(figsize=(11, 6))

    for j in range(len(x_chunk)):
        ax.vlines(x=x_chunk[j], ymin=y_bl_chunk[j], ymax=y_constr_chunk[j], 
                  color='#7f7f7f', linestyle='--', linewidth=1.5, alpha=0.8, zorder=2)

    ax.scatter(x_chunk, y_constr_chunk, color='#d62728', label='Algoritmo Construtivo (Origem)', s=70, zorder=3)
    ax.scatter(x_chunk, y_bl_chunk, color='#1f77b4', label='Melhor Busca Local (Resultado)', s=70, zorder=3)

    ax.set_xlabel('Instâncias', fontweight='bold')
    ax.set_ylabel('Valor da Função Objetivo (Número de Transações)', fontweight='bold')
    ax.set_title(f'Análise de Ganho da Busca Local a partir de sua Origem (Parte {i+1})', fontsize=13, fontweight='bold')

    plt.xticks(rotation=45, ha='right')

    ax.grid(True, linestyle=':', alpha=0.7, zorder=1)
    ax.legend(loc='upper left')

    fig.tight_layout()
    plt.savefig(f"output/evolucao_ganho_bl_parte_{i+1}.png", dpi=100)
    plt.show()